# Health Data Extraction

From Garmin watch

In [2]:
import os
import sys
import importlib
import datetime
import requests
import getpass
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pandas.core.methods.describe import reorder_columns
from sympy.abc import alpha
from torch.distributions.constraints import positive

from tqdm import tqdm
import pandas as pd
import numpy as np
from garth.exc import GarthException, GarthHTTPError
from garminconnect import (
    Garmin,
    GarminConnectAuthenticationError,
    GarminConnectConnectionError,
    GarminConnectTooManyRequestsError,
)

import lib

importlib.reload(lib)

project = lib.Project()
log = lib.getLogger(project.name)

DATE_FORMAT = '%Y-%m-%d'


# Garmin helper functions to interact with API
Taken from example file provided in documentation to ensure safe usage with API

In [2]:
def safe_api_call(api_method, *args, **kwargs):
    """
    Safe API call wrapper with comprehensive error handling.

    This demonstrates the error handling patterns used throughout the library.
    Returns (success: bool, result: Any, error_message: str)
    """
    try:
        result = api_method(*args, **kwargs)
        return True, result, None

    except GarthHTTPError as e:
        # Handle specific HTTP errors gracefully
        error_str = str(e)
        status_code = getattr(getattr(e, "response", None), "status_code", None)

        if status_code == 400 or "400" in error_str:
            return (
                False,
                None,
                "Endpoint not available (400 Bad Request) - Feature may not be enabled for your account",
            )
        elif status_code == 401 or "401" in error_str:
            return (
                False,
                None,
                "Authentication required (401 Unauthorized) - Please re-authenticate",
            )
        elif status_code == 403 or "403" in error_str:
            return (
                False,
                None,
                "Access denied (403 Forbidden) - Account may not have permission",
            )
        elif status_code == 404 or "404" in error_str:
            return (
                False,
                None,
                "Endpoint not found (404) - Feature may have been moved or removed",
            )
        elif status_code == 429 or "429" in error_str:
            return (
                False,
                None,
                "Rate limit exceeded (429) - Please wait before making more requests",
            )
        elif status_code == 500 or "500" in error_str:
            return (
                False,
                None,
                "Server error (500) - Garmin's servers are experiencing issues",
            )
        elif status_code == 503 or "503" in error_str:
            return (
                False,
                None,
                "Service unavailable (503) - Garmin's servers are temporarily unavailable",
            )
        else:
            return False, None, f"HTTP error: {e}"

    except FileNotFoundError:
        return (
            False,
            None,
            "No valid tokens found. Please login with your email/password to create new tokens.",
        )

    except GarminConnectAuthenticationError as e:
        return False, None, f"Authentication issue: {e}"

    except GarminConnectConnectionError as e:
        return False, None, f"Connection issue: {e}"

    except GarminConnectTooManyRequestsError as e:
        return False, None, f"Rate limit exceeded: {e}"

    except Exception as e:
        return False, None, f"Unexpected error: {e}"

def get_credentials():
    """Get email and password from environment or user input."""
    email = os.getenv("EMAIL")
    password = os.getenv("PASSWORD")

    if not email:
        email = input("Login email: ")
    if not password:
        password = getpass("Enter password: ")

    return email, password

def init_api() -> Garmin | None:
    """Initialize Garmin API with authentication and token management."""

    # Configure token storage
    tokenstore = os.getenv("GARMINTOKENS", "~/.garminconnect")
    tokenstore_path = Path(tokenstore).expanduser()

    print(f"🔐 Token storage: {tokenstore_path}")

    # Check if token files exist
    if tokenstore_path.exists():
        print("📄 Found existing token directory")
        token_files = list(tokenstore_path.glob("*.json"))
        if token_files:
            print(
                f"🔑 Found {len(token_files)} token file(s): {[f.name for f in token_files]}"
            )
        else:
            print("⚠️ Token directory exists but no token files found")
    else:
        print("📭 No existing token directory found")

    # First try to login with stored tokens
    try:
        print("🔄 Attempting to use saved authentication tokens...")
        garmin = Garmin()
        garmin.login(str(tokenstore_path))
        print("✅ Successfully logged in using saved tokens!")
        return garmin

    except (
        FileNotFoundError,
        GarthHTTPError,
        GarminConnectAuthenticationError,
        GarminConnectConnectionError,
    ):
        print("🔑 No valid tokens found. Requesting fresh login credentials.")

    # Loop for credential entry with retry on auth failure
    while True:
        try:
            # Get credentials
            email, password = get_credentials()

            print("� Logging in with credentials...")
            garmin = Garmin(
                email=email, password=password, is_cn=False, return_on_mfa=True
            )
            result1, result2 = garmin.login()

            if result1 == "needs_mfa":
                print("🔐 Multi-factor authentication required")

                mfa_code = input("Please enter your MFA code: ")
                print("🔄 Submitting MFA code...")

                try:
                    garmin.resume_login(result2, mfa_code)
                    print("✅ MFA authentication successful!")

                except GarthHTTPError as garth_error:
                    # Handle specific HTTP errors from MFA
                    error_str = str(garth_error)
                    if "429" in error_str and "Too Many Requests" in error_str:
                        print("❌ Too many MFA attempts")
                        print("💡 Please wait 30 minutes before trying again")
                        sys.exit(1)
                    elif "401" in error_str or "403" in error_str:
                        print("❌ Invalid MFA code")
                        print("💡 Please verify your MFA code and try again")
                        continue
                    else:
                        # Other HTTP errors - don't retry
                        print(f"❌ MFA authentication failed: {garth_error}")
                        sys.exit(1)

                except GarthException as garth_error:
                    print(f"❌ MFA authentication failed: {garth_error}")
                    print("💡 Please verify your MFA code and try again")
                    continue

            # Save tokens for future use
            garmin.garth.dump(str(tokenstore_path))
            print(f"💾 Authentication tokens saved to: {tokenstore_path}")
            print("✅ Login successful!")
            return garmin

        except GarminConnectAuthenticationError:
            print("❌ Authentication failed:")
            print("💡 Please check your username and password and try again")
            # Continue the loop to retry
            continue

        except (
            FileNotFoundError,
            GarthHTTPError,
            GarminConnectConnectionError,
            requests.exceptions.HTTPError,
        ) as err:
            print(f"❌ Connection error: {err}")
            print("💡 Please check your internet connection and try again")
            return None

        except KeyboardInterrupt:
            print("\n👋 Cancelled by user")
            return None


# Get Garmin client

In [3]:
api = init_api()

🔐 Token storage: /Users/christophermagno/.garminconnect
📄 Found existing token directory
🔑 Found 2 token file(s): ['oauth2_token.json', 'oauth1_token.json']
🔄 Attempting to use saved authentication tokens...
✅ Successfully logged in using saved tokens!


# Helper functions for datetime

In [4]:
def _get_date_string(date):
    return date.strftime(DATE_FORMAT)

def today():
    return datetime.date.today().strftime(DATE_FORMAT)


def get_date_range(start=None, rng=None):
    start = start or datetime.datetime.today()
    rng = rng or int(start.strftime('%j'))
    dates = reversed([start - datetime.timedelta(days=x) for x in range(rng)])
    return [x.strftime(DATE_FORMAT) for x in dates]


)# Helper functions to gather ando organize Garmin data

Some useful methods from the Garmin class to use
* get_stats - using
* get_steps_data
* get_daily_steps
* get_floors
* get_heart_rates - using
* get_sleep_data - using
* get_stress_data
* get_rhr_day
* get_hrv_data
* get_fitnessage_data - using

Others toook at
* get_activities
* get_activities_fordate
* get_earned_badges

In [5]:
def get_sleep_data(date):

    sleep_data = {}

    to_pop = [
        'id',
        'userProfilePK',
        'napTimeSeconds',
        'sleepWindowConfirmed',
        'sleepWindowConfirmationType',
        'autoSleepStartTimestampGMT',
        'autoSleepEndTimestampGMT',
        'sleepQualityTypePK',
        'sleepResultTypePK',
        'deviceRemCapable',
        'retro',
        'sleepFromDevice',
        'sleepScores',
        'sleepScoreInsight',
        'sleepScorePersonalizedInsight',
        'sleepVersion'
    ]

    data = safe_api_call(api.get_sleep_data, date)[1]

    sleep_data.update(data['dailySleepDTO'])
    if 'sleepScores' in sleep_data:
        sleep_data['sleepScore'] = sleep_data['sleepScores']['overall']['value']
        sleep_data['sleepScoreQuality'] = sleep_data['sleepScores']['overall']['qualifierKey']
        sleep_data['stressSleepQuality'] = sleep_data['sleepScores']['stress']['qualifierKey']
        sleep_data['awakeCountQuality'] = sleep_data['sleepScores']['awakeCount']['qualifierKey']
        sleep_data['remSleepQuality'] = sleep_data['sleepScores']['remPercentage']['qualifierKey']
        sleep_data['restlessnessSleepQuality'] = sleep_data['sleepScores']['restlessness']['qualifierKey']
        sleep_data['lightSleepQuality'] = sleep_data['sleepScores']['lightPercentage']['qualifierKey']
        sleep_data['deepSleepQuality'] = sleep_data['sleepScores']['deepPercentage']['qualifierKey']
    else:
        sleep_data['sleepScore'] = None
        sleep_data['sleepScoreQuality'] = None
        sleep_data['stressSleepQuality'] = None
        sleep_data['awakeCountQuality'] = None
        sleep_data['remSleepQuality'] = None
        sleep_data['restlessnessSleepQuality'] = None
        sleep_data['lightSleepQuality'] = None
        sleep_data['deepSleepQuality'] = None

    # result['sleepHeartRate'] = data['sleepHeartRate']
    sleep_data['avgOvernightHrv'] = data.get('avgOvernightHrv')
    sleep_data['hrvStatus'] = data.get('hrvStatus')
    sleep_data['restingHeartRate'] = data.get('restingHeartRate')

    for key in to_pop:
        try:
            sleep_data.pop(key)
        except KeyError as e:
            pass

    return sleep_data


In [6]:
def get_health_data(date):
    """
    """

    to_pop = [
        'userProfileId',
        'userDailySummaryId',
        'burnedKilocalories',
        'wellnessActiveKilocalories',
        'netRemainingKilocalories',
        'rule',
        'wellnessStartTimeGmt',
        'wellnessStartTimeLocal',
        'wellnessEndTimeGmt',
        'wellnessEndTimeLocal',
        'durationInMilliseconds',
        'wellnessDescription',
        'includesWellnessData',
        'includesActivityData',
        'includesCalorieConsumedData',
        'privacyProtected',
        'floorsAscended',
        'floorsDescended',
        'lastSevenDaysAvgRestingHeartRate',
        'source',
        'lastSyncTimestampGMT',
        'bodyBatteryMostRecentValue',
        'bodyBatteryVersion',
        'averageSpo2',
        'lowestSpo2',
        'latestSpo2',
        'latestSpo2ReadingTimeGmt',
        'latestSpo2ReadingTimeLocal',
        'latestSpo2ReadingTimeLocalaverageMonitoringEnvironmentAltitude',
        'restingCaloriesFromActivity',
        'latestRespirationValue',
        'latestRespirationTimeGMT',
        'respirationAlgorithmVersion',
        'ageGroup',
        'averageMonitoringEnvironmentAltitude',
        'bodyBatteryChargedValue',
        'bodyBatteryDrainedValue',
        'bodyBatteryHighestValue',
        'bodyBatteryLowestValue',
        'bodyBatteryDuringSleep',
        'wellnessKilocalories',
        'consumedKilocalories',
        'remainingKilocalories',
        'netCalorieGoal',
        'wellnessDistanceMeters',
        'userNote',
        'sleepingSeconds',
        'minAvgHeartRate',
        'maxAvgHeartRate',
        'abnormalHeartRateAlertsCount',
        'unmeasurableSleepSeconds',
        'measurableAsleepDuration',
        'measurableAwakeDuration',
        'stressPercentage',
        'restStressPercentage',
        'activityStressPercentage',
        'uncategorizedStressPercentage',
        'lowStressPercentage',
        'mediumStressPercentage',
        'highStressPercentage',
        'restStressDuration',
        'userFloorsAscendedGoal',
    ]

    health_data = safe_api_call(api.get_stats, date)[1]
    health_data['fitnessAge'] = int(safe_api_call(api.get_fitnessage_data, date)[1]['fitnessAge'])

    # Heart rate data
    hdata = safe_api_call(api.get_heart_rates, date)[1]['heartRateValues']
    if hdata:
        heart_rates = [v[1] for v in hdata if v[1] is not None]
        health_data['avgHeartRate'] = float(np.array(heart_rates).mean())

    health_data.update(get_sleep_data(date))

    for key in to_pop:
        try:
            health_data.pop(key)
        except KeyError as e:
            pass

    health_data_clean = {}
    for key, value in health_data.items():
        health_data_clean[lib.convert_camel_case(key).title()] = value

    return health_data_clean.pop('Uuid'), health_data_clean

# Get Health Data

In [7]:
dates = get_date_range()

In [8]:
overall_data = {}
for date in tqdm(dates):
    id, health_data = get_health_data(date)
    overall_data[id] = health_data


100%|██████████| 354/354 [02:36<00:00,  2.26it/s]


# Create the Health Dataframe

In [9]:
df = pd.DataFrame(overall_data).T.convert_dtypes()

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 354 entries, 8f2ddf09f3e04ac780a5470c4099a803 to c66a8141f18c4da98d70cb9e7c784fd8
Data columns (total 57 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Total Kilocalories             354 non-null    Int64  
 1   Active Kilocalories            354 non-null    Int64  
 2   Bmr Kilocalories               354 non-null    Int64  
 3   Total Steps                    353 non-null    Int64  
 4   Total Distance Meters          353 non-null    Int64  
 5   Calendar Date                  354 non-null    string 
 6   Daily Step Goal                354 non-null    Int64  
 7   Highly Active Seconds          354 non-null    Int64  
 8   Active Seconds                 354 non-null    Int64  
 9   Sedentary Seconds              354 non-null    Int64  
 10  Moderate Intensity Minutes     354 non-null    Int64  
 11  Vigorous Intensity Minutes     354 non-null    Int64  


In [11]:
df.columns.tolist()

['Total Kilocalories',
 'Active Kilocalories',
 'Bmr Kilocalories',
 'Total Steps',
 'Total Distance Meters',
 'Calendar Date',
 'Daily Step Goal',
 'Highly Active Seconds',
 'Active Seconds',
 'Sedentary Seconds',
 'Moderate Intensity Minutes',
 'Vigorous Intensity Minutes',
 'Floors Ascended In Meters',
 'Floors Descended In Meters',
 'Intensity Minutes Goal',
 'Min Heart Rate',
 'Max Heart Rate',
 'Resting Heart Rate',
 'Average Stress Level',
 'Max Stress Level',
 'Stress Duration',
 'Activity Stress Duration',
 'Uncategorized Stress Duration',
 'Total Stress Duration',
 'Low Stress Duration',
 'Medium Stress Duration',
 'High Stress Duration',
 'Stress Qualifier',
 'Body Battery At Wake Time',
 'Avg Waking Respiration Value',
 'Highest Respiration Value',
 'Lowest Respiration Value',
 'Fitness Age',
 'Sleep Time Seconds',
 'Sleep Start Timestamp G M T',
 'Sleep End Timestamp G M T',
 'Sleep Start Timestamp Local',
 'Sleep End Timestamp Local',
 'Deep Sleep Seconds',
 'Light Sleep 

In [12]:
remap_columns = {
    'Calendar Date': 'Date',

    'Fitness Age': 'Fitness Age',

    # Calories
    'Total Kilocalories': 'Calories',
    'Active Kilocalories': 'Active Calories',
    'Bmr Kilocalories': 'Resting Calories',

    # Heart Rate
    'Avg Heart Rate': 'Average Heart Rate',
    'Min Heart Rate': 'Min Heart Rate',
    'Max Heart Rate': 'Max Heart Rate',
    'Resting Heart Rate': 'Resting Heart Rate',
    'Hrv Status': 'Heart Rate Variability Qualifier',

    # Respiration
    'Avg Waking Respiration Value': 'Avg Waking Respiration Value',
    'Highest Respiration Value': 'Highest Respiration Value',
    'Lowest Respiration Value': 'Lowest Respiration Value',

    # Steps/Distance
    'Total Steps': 'Total Steps',
    'Total Distance Meters': 'Total Distance Meters',
    'Daily Step Goal': 'Daily Step Goal',

    # Floors
    'Floors Ascended In Meters': 'Floors Ascended In Meters',
    'Floors Descended In Meters': 'Floors Descended In Meters',

    # Activity
    'Active Seconds': 'Active Seconds',
    'Highly Active Seconds': 'Highly Active Seconds',
    'Sedentary Seconds': 'Sedentary Seconds',
    'Moderate Intensity Minutes': 'Moderate Intensity Minutes',
    'Vigorous Intensity Minutes': 'Vigorous Intensity Minutes',
    'Intensity Minutes Goal': 'Intensity Minutes Goal',

    # Stress
    'Average Stress Level': 'Average Stress Level',
    'Total Stress Duration': 'Total Stress Seconds',
    'Stress Duration': 'Stress Seconds',
    'Max Stress Level': 'Max Stress Level',
    'Uncategorized Stress Duration': 'Uncategorized Stress Seconds',
    'Low Stress Duration': 'Low Stress Seconds',
    'Medium Stress Duration': 'Medium Stress Seconds',
    'High Stress Duration': 'High Stress Seconds',
    'Activity Stress Duration': 'Activity Stress Seconds',
    'Stress Qualifier': 'Stress Qualifier',

    # Body battery
    'Body Battery At Wake Time': 'Body Battery',

    # Sleep data
    'Sleep Start Timestamp G M T': 'Sleep Start Timestamp GMT',
    'Sleep End Timestamp G M T': 'Sleep End Timestamp GMT',
    'Sleep Start Timestamp Local': 'Sleep Start Timestamp Local',
    'Sleep End Timestamp Local': 'Sleep End Timestamp Local',

    'Sleep Time Seconds': 'Sleep Time Seconds',
    'Sleep Score': 'Sleep Score',
    'Sleep Score Quality': 'Sleep Quality',
    'Sleep Score Feedback': 'Sleep Feedback',

    'Light Sleep Seconds': 'Light Sleep Seconds',
    'Deep Sleep Seconds': 'Deep Sleep Seconds',
    'Rem Sleep Seconds': 'Rem Sleep Seconds',
    'Awake Sleep Seconds': 'Awake Sleep Seconds',

    'Average Respiration Value': 'Average Respiration Value',
    'Awake Count': 'Awake Count',
    'Avg Sleep Stress': 'Avg Sleep Stress',

    'Stress Sleep Quality': 'Stress Sleep Quality',
    'Awake Count Quality': 'Awake Count Quality',
    'Rem Sleep Quality': 'Rem Sleep Quality',
    'Restlessness Sleep Quality': 'Restlessness Sleep Quality',
    'Light Sleep Quality': 'Light Sleep Quality',
    'Deep Sleep Quality': 'Deep Sleep Quality',

    'Avg Overnight Hrv': 'Average Overnight Hrv',
}

In [13]:
df = df[remap_columns.keys()]
df = df.rename(columns=remap_columns)
df

,Date,Fitness Age,Calories,Active Calories,Resting Calories,Average Heart Rate,Min Heart Rate,Max Heart Rate,Resting Heart Rate,Heart Rate Variability Qualifier,...,Average Respiration Value,Awake Count,Avg Sleep Stress,Stress Sleep Quality,Awake Count Quality,Rem Sleep Quality,Restlessness Sleep Quality,Light Sleep Quality,Deep Sleep Quality,Average Overnight Hrv
8f2ddf09f3e04ac780a5470c4099a803,2025-01-01,30,2322,833,1489,<NA>,48,139,50,<NA>,...,13,2,11,EXCELLENT,FAIR,EXCELLENT,FAIR,FAIR,FAIR,<NA>
4ab88cfdf283481fa593979341aa8332,2025-01-02,30,1960,471,1489,<NA>,53,131,55,<NA>,...,13,0,35,POOR,EXCELLENT,FAIR,EXCELLENT,EXCELLENT,EXCELLENT,<NA>
a6433536e16a492e8a7655909869ea9c,2025-01-03,30,1878,389,1489,<NA>,49,130,52,<NA>,...,13,3,14,GOOD,FAIR,EXCELLENT,FAIR,GOOD,FAIR,<NA>
8709a878c8b8459f8769c1ff58d886d3,2025-01-04,30,2388,899,1489,<NA>,49,142,52,<NA>,...,13,2,17,FAIR,FAIR,FAIR,FAIR,FAIR,EXCELLENT,<NA>
e929e59be1114b58a17d010d389233ed,2025-01-05,30,2536,1047,1489,<NA>,59,142,57,<NA>,...,14,4,41,POOR,POOR,POOR,POOR,POOR,FAIR,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3c68feb2861848a6884e16690bed0fff,2025-12-16,31,1927,410,1517,72.175362,49,122,51,BALANCED,...,13,0,12,EXCELLENT,EXCELLENT,FAIR,EXCELLENT,EXCELLENT,FAIR,64
b66ae2c8866943f09ee44c9c702e6464,2025-12-17,31,1791,274,1517,70.95,49,113,52,BALANCED,...,14,0,12,EXCELLENT,EXCELLENT,GOOD,EXCELLENT,GOOD,EXCELLENT,58
0249e5d497ac4ce18d6e9896606ee2fa,2025-12-18,30,3154,1637,1517,83.961111,51,147,54,BALANCED,...,15,1,17,FAIR,EXCELLENT,FAIR,EXCELLENT,EXCELLENT,EXCELLENT,54
5f3d7033f28b4c96ad99f91a13cdde7e,2025-12-19,30,3092,1575,1517,83.591667,49,154,51,BALANCED,...,13,0,20,FAIR,EXCELLENT,GOOD,EXCELLENT,EXCELLENT,EXCELLENT,55


# Convert some types

In [14]:
for col in ['Sleep Start Timestamp GMT', 'Sleep End Timestamp GMT', 'Sleep Start Timestamp Local', 'Sleep End Timestamp Local']:
    df[col] = pd.to_datetime(df[col], unit='ms')

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 354 entries, 8f2ddf09f3e04ac780a5470c4099a803 to c66a8141f18c4da98d70cb9e7c784fd8
Data columns (total 57 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   Date                              354 non-null    string        
 1   Fitness Age                       354 non-null    Int64         
 2   Calories                          354 non-null    Int64         
 3   Active Calories                   354 non-null    Int64         
 4   Resting Calories                  354 non-null    Int64         
 5   Average Heart Rate                142 non-null    Float64       
 6   Min Heart Rate                    354 non-null    Int64         
 7   Max Heart Rate                    354 non-null    Int64         
 8   Resting Heart Rate                345 non-null    Int64         
 9   Heart Rate Variability Qualifier  132 non-null    string        


# Export the data

In [16]:
df.to_csv(project.raw_file)